In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import pandas as pd
import os

train_dir = ""
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train' in dirs:
        train_dir = os.path.join(root, 'train')
        break

base_dir = os.path.dirname(train_dir)
test_dir = os.path.join(base_dir, 'test')

print("1. Chuan bi data tu:", train_dir)
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.2, subset="training", seed=42, image_size=(32, 32), batch_size=32)
val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.2, subset="validation", seed=42, image_size=(32, 32), batch_size=32)

print("2. Build model...")
model = models.Sequential([
    layers.InputLayer(input_shape=(32, 32, 3)),
    layers.Rescaling(1./255),
    
    layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("3. Train model...")
model.fit(train_ds, validation_data=val_ds, epochs=15)

print("4. Convert tflite...")
cvt_f32 = tf.lite.TFLiteConverter.from_keras_model(model)
with open('model_float32.tflite', 'wb') as f:
    f.write(cvt_f32.convert())

cvt_f16 = tf.lite.TFLiteConverter.from_keras_model(model)
cvt_f16.optimizations = [tf.lite.Optimize.DEFAULT]
cvt_f16.target_spec.supported_types = [tf.float16]
with open('model_float16.tflite', 'wb') as f:
    f.write(cvt_f16.convert())

def rep_data_gen():
    for img, _ in train_ds.unbatch().batch(1).take(100):
        yield [img]

cvt_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
cvt_int8.optimizations = [tf.lite.Optimize.DEFAULT]
cvt_int8.representative_dataset = rep_data_gen
cvt_int8.target_spec.supported_ops =[tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
cvt_int8.inference_input_type = tf.int8
cvt_int8.inference_output_type = tf.int8
with open('model_int8.tflite', 'wb') as f:
    f.write(cvt_int8.convert())

print("5. Predict de tao file submit...")
interpreter = tf.lite.Interpreter(model_path="model_int8.tflite")
interpreter.allocate_tensors()
in_idx = interpreter.get_input_details()[0]['index']
out_idx = interpreter.get_output_details()[0]['index']
scale, zp = interpreter.get_input_details()[0]["quantization"]
dtype = interpreter.get_input_details()[0]["dtype"]

ket_qua =[]
danh_sach_anh = sorted(os.listdir(test_dir))

for ten_anh in danh_sach_anh:
    duong_dan = os.path.join(test_dir, ten_anh)
    img = tf.keras.utils.load_img(duong_dan, target_size=(32, 32))
    img_arr = tf.keras.utils.img_to_array(img) / 255.0
    
    img_arr = img_arr / scale + zp
    img_arr = np.expand_dims(img_arr, axis=0).astype(dtype)
    
    interpreter.set_tensor(in_idx, img_arr)
    interpreter.invoke()
    
    pred = np.argmax(interpreter.get_tensor(out_idx)[0])
    id_anh = ten_anh.replace('.png', '')
    ket_qua.append([id_anh, pred])

pd.DataFrame(ket_qua, columns=['Id', 'Label']).to_csv('submission.csv', index=False)
print("Xong het roi!")

2026-05-09 16:35:00.796414: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778344501.060738      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778344501.139302      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778344501.763612      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778344501.763662      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778344501.763666      16 computation_placer.cc:177] computation placer alr

1. Chuan bi data tu: /kaggle/input/competitions/edge-ai-challenge-2026-qualifiers/train
Found 9629 files belonging to 10 classes.
Using 7704 files for training.


2026-05-09 16:35:45.793353: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Found 9629 files belonging to 10 classes.
Using 1925 files for validation.
2. Build model...
3. Train model...
Epoch 1/15


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


...

INFO:tensorflow:Assets written to: /tmp/tmp2n747in_/assets


INFO:tensorflow:Assets written to: /tmp/tmp2n747in_/assets


Saved artifact at '/tmp/tmp2n747in_'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 32, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  136077684976400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136077684979280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136077684979088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136077684976208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136077684979664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136077684980240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136077684977744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136077684981584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136077684981008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136077684980816: TensorSpec(shape=(), dtype=tf.resource, name=None)


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1778344681.699423      16 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1778344681.699451      16 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


5. Predict de tao file submit...
Xong het roi!


In [2]:
!xxd -i model_int8.tflite > model.cc